In [2]:
import os
import sys
import sqlite3
from pathlib import Path
import importlib.util

# ===== 共通ユーティリティ =====

def find_config_py(start: Path, max_up: int = 6) -> Path | None:
    """start から最大 max_up 階層まで上に遡って utils/config.py を探す"""
    cur = start.resolve()

    for _ in range(max_up + 1):
        candidate = cur / "utils" / "config.py"

        if candidate.is_file():
            return candidate

        cur = cur.parent

    return None


def import_config_from_path(cfg_path: Path):
    """ファイルパスから utils.config を読み込む"""
    spec = importlib.util.spec_from_file_location(
        "utils.config",
        cfg_path
    )

    if spec and spec.loader:
        module = importlib.util.module_from_spec(spec)
        sys.modules["utils.config"] = module
        spec.loader.exec_module(module)
        return module

    raise ImportError(
        f"Failed to import config from: {cfg_path}"
    )


# ===== ベースディレクトリ決定 =====

try:
    base_dir = Path(__file__).resolve().parent

except NameError:
    base_dir = Path.cwd()


# ===== config 解決 =====

config_module = None

cfg_path = find_config_py(base_dir)


if cfg_path:

    config_module = import_config_from_path(cfg_path)


else:

    sys.path.append(
        str((base_dir / "..").resolve())
    )

    try:
        from utils.config import PROJECT_DIR  # type: ignore

    except Exception:

        PROJECT_DIR = None

    else:

        config_module = sys.modules.get(
            "utils.config"
        )


# ===== PROJECT_DIR取得 =====

if config_module is None:

    PROJECT_DIR = os.getenv(
        "PROJECT_DIR",
        base_dir.name
    )

else:

    PROJECT_DIR = getattr(
        config_module,
        "PROJECT_DIR",
        os.getenv(
            "PROJECT_DIR",
            base_dir.name
        )
    )


# ===== DBパス =====

if os.name == "nt":

    home = Path(
        os.environ.get(
            "USERPROFILE",
            str(Path.home())
        )
    )

else:

    home = Path.home()


user_base = home / "myenv310" / PROJECT_DIR

db_dir = user_base / "db"

db_dir.mkdir(
    parents=True,
    exist_ok=True
)


db_path = db_dir / "output.db"

columns_file = db_dir / "columns_product_summary.txt"


# ===== columnsファイル探索 =====

if not columns_file.exists():

    project_root = (
        cfg_path.parent.parent
        if cfg_path
        else base_dir
    )

    alt_candidates = [

        project_root
        / "db"
        / "columns_product_summary.txt",

        base_dir
        / "db"
        / "columns_product_summary.txt",

        Path.cwd()
        / "db"
        / "columns_product_summary.txt"

    ]

    for c in alt_candidates:

        if c.exists():

            columns_file = c
            break



# ===== カラム定義読み込み =====

if not columns_file.exists():

    raise FileNotFoundError(
        f"columns_product_summary.txt が見つかりません: {columns_file}"
    )


with columns_file.open(
    encoding="utf-8"
) as f:

    cols_types = [
        line.strip()
        for line in f
        if line.strip()
    ]


if not cols_types:

    raise ValueError(
        "columns_product_summary.txt が空です"
    )


columns_definitions = ",\n    ".join(
    cols_types
)


# ===== テーブル名 =====

table_name = "product_summary"



# ===== CREATE TABLE =====

create_table_sql = f"""

CREATE TABLE IF NOT EXISTS {table_name} (

    {columns_definitions}

);

"""


print("=" * 80)
print("[INFO] CREATE TABLE SQL")
print("=" * 80)

print(create_table_sql)



# ===== DB作成 =====

conn = sqlite3.connect(
    str(db_path)
)


try:

    cur = conn.cursor()

    cur.executescript(
        create_table_sql
    )

    conn.commit()


finally:

    conn.close()



print("=" * 80)
print("[INFO] ✅ テーブル作成完了")
print(f"[INFO] DB : {db_path}")
print(f"[INFO] TABLE : {table_name}")
print(f"[INFO] PROJECT_DIR : {PROJECT_DIR}")
print(f"[INFO] COLUMN FILE : {columns_file}")
print("=" * 80)

[INFO] CREATE TABLE SQL


CREATE TABLE IF NOT EXISTS product_summary (

    id INTEGER PRIMARY KEY AUTOINCREMENT,
    master_machine_id INTEGER UNIQUE,
    master_machine_name TEXT,
    master_machine_maker TEXT,
    master_machine_model TEXT,
    master_machine_type TEXT,
    master_machine_gouki TEXT,
    master_machine_memo TEXT,
    master_machine_dis TEXT,
    master_machine_pworld_url TEXT,
    master_machine_pworldimage_url TEXT,
    latest_price INTEGER,
    min_price INTEGER,
    max_price INTEGER,
    avg_price REAL,
    median_price INTEGER,
    price_count INTEGER,
    shop_count INTEGER,
    lowest_shop_name TEXT,
    lowest_product_url TEXT,
    first_seen DATETIME,
    last_seen DATETIME,
    latest_scraped_at DATETIME,
    created_at DATETIME,
    updated_at DATETIME

);


[INFO] ✅ テーブル作成完了
[INFO] DB : C:\Users\Owner\myenv310\soubanavi-s\db\output.db
[INFO] TABLE : product_summary
[INFO] PROJECT_DIR : soubanavi-s
[INFO] COLUMN FILE : C:\Users\Owner\myenv310\soubanavi-s\